# HBO Max · Películas y Series
**Actividad 4 · Mi primer prototipo de dashboard** — Visualización de datos, Universidad de La Salle.

**Dataset:** catálogo de HBO Max en EE. UU. (JustWatch, mayo de 2022, Kaggle): `titles.csv` con más de 3.000 títulos
y `credits.csv` con más de 66.000 créditos de actores y directores.

**Preguntas que responde este análisis:**

1. ¿HBO Max es un catálogo de estrenos o de clásicos?
2. ¿Cómo ha cambiado la oferta de géneros con el tiempo?
3. ¿Los géneros más abundantes son también los mejor valorados?
4. ¿Las producciones antiguas están mejor valoradas que las recientes?
5. ¿Las películas se han vuelto más largas o más cortas?
6. ¿Lo más popular es lo mejor valorado?
7. ¿Qué es lo mejor del catálogo y de dónde viene?
8. ¿Las series que duran más temporadas son mejores?
9. ¿Qué directores definen el catálogo y qué tan bien valorados están?
10. ¿Quiénes aparecen más en el catálogo y en qué tipo de contenido?

**Arquitectura.** El código es un paquete de Python separado en tres capas:
- `data`: lectura y limpieza de los CSV (único punto que toca el disco).
- `logic`: filtros y métricas como funciones puras sobre DataFrames (sin Dash ni Plotly).
- `views`: catálogo de gráficas, componentes, layout y callbacks de Dash.

El mismo catálogo de gráficas alimenta el análisis estático de este notebook y el dashboard interactivo.

## 1. Dependencias

In [ ]:
%pip install -q dash plotly pandas

## 2. Datos
Si no están en `archive/` ni en `/content`, se piden con el diálogo de carga de Colab.

In [ ]:
from pathlib import Path

DATA_DIR = next((p for p in (Path("archive"), Path("/content")) if (p / "titles.csv").exists()), None)
if DATA_DIR is None:
    from google.colab import files  # solo disponible en Colab

    files.upload()  # seleccionar titles.csv y credits.csv
    DATA_DIR = Path(".")
print("Datos en:", DATA_DIR.resolve())

## 3. Código del proyecto
Cada celda escribe un módulo del paquete `hbo_max_dashboard`.

In [ ]:
from pathlib import Path

for sub in ('data', 'logic', 'views'):
    Path("hbo_max_dashboard", sub).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile hbo_max_dashboard/__init__.py
"""Dashboard interactivo del catálogo de HBO Max.

Capas (la dependencia solo apunta hacia abajo):
    views  ->  logic      (Dash/Plotly; filtros y métricas)
    app    ->  data       (composición; lectura y limpieza de CSV)
"""

In [ ]:
%%writefile hbo_max_dashboard/app.py
"""Composición: carga los datos, arma el layout y registra los callbacks."""

from pathlib import Path

from dash import Dash

from .config import DATA_DIR
from .data import load_catalog
from .logic.filters import genre_options, year_bounds
from .views.callbacks import register_callbacks
from .views.layout import build_layout


def create_app(data_dir: Path = DATA_DIR) -> Dash:
    catalog = load_catalog(data_dir)
    app = Dash(__name__, title="HBO Max · Películas y Series")
    app.layout = build_layout(genre_options(catalog.titles), year_bounds(catalog.titles))
    register_callbacks(app, catalog)
    return app

In [ ]:
%%writefile hbo_max_dashboard/config.py
"""Constantes de dominio compartidas por todas las capas."""

from pathlib import Path

DATA_DIR = Path(__file__).resolve().parents[2] / "archive"

MOVIE, SHOW = "Película", "Serie"
TYPE_LABELS = {"MOVIE": MOVIE, "SHOW": SHOW}

GENRE_LABELS = {
    "action": "acción",
    "animation": "animación",
    "comedy": "comedia",
    "crime": "crimen",
    "documentation": "documental",
    "drama": "drama",
    "european": "europeo",
    "family": "familiar",
    "fantasy": "fantasía",
    "history": "historia",
    "horror": "terror",
    "music": "música",
    "reality": "reality",
    "romance": "romance",
    "scifi": "ciencia ficción",
    "sport": "deporte",
    "thriller": "suspenso",
    "war": "bélico",
    "western": "western",
}

ALL_GENRES = "__all__"
TOP_N = 10
MIN_GROUP_SIZE = 10  # tamaño mínimo de un grupo para que su promedio sea confiable
MIN_DECADE_SIZE = 20  # décadas con menos títulos se omiten en las series temporales
MIN_VOTES = 1_000  # votos mínimos para usar la puntuación IMDb de un título
TOP_RATED_MIN_VOTES = 10_000

In [ ]:
%%writefile hbo_max_dashboard/data/__init__.py
"""Capa de datos: lectura y limpieza. No conoce la lógica de negocio ni la UI."""

from .catalog import Catalog
from .repository import load_catalog

__all__ = ["Catalog", "load_catalog"]

In [ ]:
%%writefile hbo_max_dashboard/data/catalog.py
"""Modelo de datos del catálogo."""

from dataclasses import dataclass

import pandas as pd


@dataclass(frozen=True)
class Catalog:
    """Títulos y créditos ya limpios. Se carga una vez y no se modifica."""

    titles: pd.DataFrame
    credits: pd.DataFrame

In [ ]:
%%writefile hbo_max_dashboard/data/cleaning.py
"""Limpieza y normalización de los datos crudos."""

import ast

import pandas as pd

from ..config import GENRE_LABELS, TYPE_LABELS


def _to_list(text: object) -> list[str]:
    """Convierte "['drama', 'crime']" en ['drama', 'crime']."""
    try:
        return ast.literal_eval(text) if isinstance(text, str) else []
    except (ValueError, SyntaxError):
        return []


def clean_titles(raw: pd.DataFrame) -> pd.DataFrame:
    titles = raw.drop_duplicates("id").copy()
    titles["genres"] = titles["genres"].apply(lambda text: [GENRE_LABELS.get(g, g) for g in _to_list(text)])
    titles["production_countries"] = titles["production_countries"].apply(_to_list)
    titles["tipo"] = titles["type"].map(TYPE_LABELS)
    titles["decada"] = titles["release_year"] // 10 * 10
    titles["age_certification"] = titles["age_certification"].fillna("Sin dato")
    return titles.reset_index(drop=True)


def clean_credits(raw: pd.DataFrame) -> pd.DataFrame:
    return raw.drop_duplicates(["person_id", "id", "role"]).reset_index(drop=True)


def missing_report(raw: pd.DataFrame) -> pd.DataFrame:
    """Valores faltantes por columna, para documentar el procesamiento."""
    missing = raw.isna().sum()
    report = pd.DataFrame({"faltantes": missing, "porcentaje": (missing / len(raw) * 100).round(1)})
    return report[report["faltantes"] > 0].sort_values("faltantes", ascending=False)

In [ ]:
%%writefile hbo_max_dashboard/data/repository.py
"""Acceso a las fuentes de datos (CSV). Único módulo que toca el disco."""

from pathlib import Path

import pandas as pd

from ..config import DATA_DIR
from .catalog import Catalog
from .cleaning import clean_credits, clean_titles


def read_csv(name: str, data_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Lee `name.csv` o `name_csv.zip`, lo que exista."""
    for candidate in (f"{name}.csv", f"{name}_csv.zip"):
        path = Path(data_dir) / candidate
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"No se encontró {name}.csv ni {name}_csv.zip en {data_dir}")


def load_catalog(data_dir: Path = DATA_DIR) -> Catalog:
    return Catalog(
        titles=clean_titles(read_csv("titles", data_dir)),
        credits=clean_credits(read_csv("credits", data_dir)),
    )

In [ ]:
%%writefile hbo_max_dashboard/logic/__init__.py
"""Capa de lógica: filtros y métricas puras sobre DataFrames. No importa Dash ni Plotly."""

In [ ]:
%%writefile hbo_max_dashboard/logic/filters.py
"""Reglas de selección: qué títulos entran según los filtros del usuario."""

from collections.abc import Sequence
from dataclasses import dataclass

import pandas as pd

from ..config import ALL_GENRES


@dataclass(frozen=True)
class Filters:
    types: tuple[str, ...]
    genre: str = ALL_GENRES
    years: tuple[int, int] | None = None

    @classmethod
    def from_inputs(cls, types: Sequence[str] | None, genre: str | None, years: Sequence[int] | None) -> "Filters":
        """Normaliza los valores crudos que llegan de los controles de la UI."""
        return cls(
            types=tuple(types or ()),
            genre=genre or ALL_GENRES,
            years=(int(years[0]), int(years[1])) if years else None,
        )

    def apply(self, titles: pd.DataFrame) -> pd.DataFrame:
        mask = titles["tipo"].isin(self.types)
        if self.years:
            mask &= titles["release_year"].between(*self.years)
        if self.genre != ALL_GENRES:
            mask &= titles["genres"].apply(lambda genres: self.genre in genres)
        return titles[mask]


def genre_options(titles: pd.DataFrame) -> list[str]:
    return sorted(titles["genres"].explode().dropna().unique())


def year_bounds(titles: pd.DataFrame) -> tuple[int, int]:
    return int(titles["release_year"].min()), int(titles["release_year"].max())

In [ ]:
%%writefile hbo_max_dashboard/logic/metrics.py
"""Indicadores y agregaciones sobre una selección de títulos.

Funciones puras: reciben DataFrames y devuelven DataFrames o valores, sin efectos.
"""

from dataclasses import dataclass

import pandas as pd

from ..config import MIN_DECADE_SIZE, MIN_GROUP_SIZE, MIN_VOTES, MOVIE, SHOW, TOP_N, TOP_RATED_MIN_VOTES


@dataclass(frozen=True)
class Kpis:
    total: int
    movies: int
    shows: int
    imdb_mean: float | None
    share_before_2000: float | None


def kpis(df: pd.DataFrame) -> Kpis:
    total = len(df)
    imdb = df["imdb_score"].mean()
    return Kpis(
        total=total,
        movies=int((df["tipo"] == MOVIE).sum()),
        shows=int((df["tipo"] == SHOW).sum()),
        imdb_mean=None if pd.isna(imdb) else float(imdb),
        share_before_2000=float((df["release_year"] < 2000).mean()) if total else None,
    )


def _rated(df: pd.DataFrame, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    """Títulos con suficientes votos para que su puntuación IMDb sea representativa."""
    return df[df["imdb_votes"] >= min_votes].dropna(subset=["imdb_score"])


def titles_by_decade(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(["decada", "tipo"]).size().reset_index(name="titulos")


def genre_share_by_decade(df: pd.DataFrame, n: int = TOP_N) -> pd.DataFrame:
    """% de títulos de cada década que pertenecen a cada uno de los `n` géneros principales."""
    per_decade = df.groupby("decada").size()
    decades = per_decade[per_decade >= MIN_DECADE_SIZE].index
    exploded = df[df["decada"].isin(decades)].explode("genres").dropna(subset=["genres"])
    top = exploded["genres"].value_counts().head(n).index
    counts = exploded[exploded["genres"].isin(top)].groupby(["genres", "decada"]).size().unstack(fill_value=0)
    return (counts.div(per_decade[decades], axis=1) * 100).round(1).reindex(top)


def genre_quality(df: pd.DataFrame, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    """Por género: cuántos títulos hay y qué tan bien valorados están."""
    exploded = df.explode("genres").dropna(subset=["genres"])
    size = exploded.groupby("genres").size().rename("titulos")
    rated = _rated(exploded, min_votes).groupby("genres")["imdb_score"].agg(imdb="mean", calificados="size")
    stats = rated.join(size)
    stats = stats[stats["calificados"] >= MIN_GROUP_SIZE]
    return stats.round({"imdb": 2}).rename_axis("genero").reset_index()


def imdb_by_decade(df: pd.DataFrame, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    stats = _rated(df, min_votes).groupby("decada")["imdb_score"].agg(imdb="mean", titulos="size")
    return stats[stats["titulos"] >= MIN_DECADE_SIZE].round({"imdb": 2}).reset_index()


def movie_runtime_by_decade(df: pd.DataFrame) -> pd.DataFrame:
    movies = df[(df["tipo"] == MOVIE) & (df["runtime"] > 0)]
    stats = movies.groupby("decada")["runtime"].agg(duracion="median", titulos="size")
    return stats[stats["titulos"] >= MIN_DECADE_SIZE].reset_index()


def popularity_vs_quality(df: pd.DataFrame, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    rated = _rated(df, min_votes)
    return rated[rated["tmdb_popularity"] > 0]


def rank_correlation(df: pd.DataFrame, x: str, y: str) -> float | None:
    """Correlación de Spearman: robusta a valores extremos como la popularidad."""
    if len(df) < MIN_GROUP_SIZE:
        return None
    return float(df[x].rank().corr(df[y].rank()))


def top_rated(df: pd.DataFrame, n: int = TOP_N, min_votes: int = TOP_RATED_MIN_VOTES) -> pd.DataFrame:
    cols = ["title", "release_year", "tipo", "imdb_score", "imdb_votes"]
    return _rated(df, min_votes).nlargest(n, ["imdb_score", "imdb_votes"])[cols]


def imdb_by_seasons(df: pd.DataFrame, max_bucket: int = 6, min_votes: int = MIN_VOTES) -> pd.DataFrame:
    """Puntuación de cada serie agrupada por número de temporadas (el último grupo es `max_bucket`+)."""
    shows = _rated(df[df["tipo"] == SHOW], min_votes).dropna(subset=["seasons"])
    seasons = shows["seasons"].astype(int).clip(upper=max_bucket)
    labels = seasons.astype(str).where(seasons < max_bucket, f"{max_bucket}+")
    return shows.assign(temporadas=labels)[["title", "temporadas", "imdb_score"]].sort_values(
        "temporadas", key=lambda s: s.str.rstrip("+").astype(int)
    )


def top_people(df: pd.DataFrame, credits: pd.DataFrame, role: str, n: int = TOP_N) -> pd.DataFrame:
    """Personas con más títulos en la selección, con su IMDb promedio y % de títulos animados."""
    sel = credits[(credits["role"] == role) & credits["id"].isin(df["id"])]
    if sel.empty:
        return pd.DataFrame(columns=["name", "titulos", "imdb", "pct_animacion"])
    info = df.set_index("id")
    sel = sel.assign(
        imdb=sel["id"].map(info["imdb_score"]),
        animada=sel["id"].map(info["genres"].apply(lambda g: "animación" in g)),
    )
    stats = sel.groupby("name").agg(titulos=("id", "nunique"), imdb=("imdb", "mean"), pct_animacion=("animada", "mean"))
    stats["pct_animacion"] *= 100
    return stats.nlargest(n, "titulos").round({"imdb": 2, "pct_animacion": 0}).reset_index()

In [ ]:
%%writefile hbo_max_dashboard/views/__init__.py
"""Capa de vistas: catálogo de gráficas, componentes, layout y callbacks de Dash."""

In [ ]:
%%writefile hbo_max_dashboard/views/callbacks.py
"""Controlador: traduce los filtros de la UI en KPIs y figuras."""

from dash import Dash, Input, Output

from ..data import Catalog
from ..logic import metrics
from ..logic.filters import Filters
from .charts import CHARTS
from .kpis import KPIS
from .layout import FILTER_GENRE, FILTER_TYPE, FILTER_YEARS


def register_callbacks(app: Dash, catalog: Catalog) -> None:
    @app.callback(
        [Output(spec.id, "children") for spec in KPIS] + [Output(spec.id, "figure") for spec in CHARTS],
        Input(FILTER_TYPE, "value"),
        Input(FILTER_GENRE, "value"),
        Input(FILTER_YEARS, "value"),
    )
    def update(types, genre, years):
        selection = Filters.from_inputs(types, genre, years).apply(catalog.titles)
        values = metrics.kpis(selection)
        return [spec.format(values) for spec in KPIS] + [spec.render(selection, catalog.credits) for spec in CHARTS]

In [ ]:
%%writefile hbo_max_dashboard/views/charts.py
"""Catálogo de visualizaciones: única fuente de verdad para el dashboard y el notebook.

Cada `ChartSpec` une una pregunta de análisis con la métrica que la responde (capa de lógica)
y la primitiva que la dibuja (`figures`). El layout, los callbacks y el notebook iteran
`CHARTS`, así que agregar una gráfica es agregar una entrada aquí.
"""

from collections.abc import Callable
from dataclasses import dataclass

import pandas as pd
import plotly.graph_objects as go

from ..logic import metrics
from . import figures

Builder = Callable[[pd.DataFrame, pd.DataFrame], go.Figure]


@dataclass(frozen=True)
class ChartSpec:
    id: str
    title: str
    question: str
    insight: str  # lectura sobre el catálogo completo, se usa en el notebook
    build: Builder

    def render(self, titles: pd.DataFrame, credits: pd.DataFrame) -> go.Figure:
        return self.build(titles, credits).update_layout(title_text=self.title)


def _catalog_by_decade(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.stacked_bar(
        metrics.titles_by_decade(titles),
        x="decada",
        y="titulos",
        color="tipo",
        labels={"decada": "Década de estreno", "titulos": "Títulos"},
    )


def _genre_mix(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.heatmap(metrics.genre_share_by_decade(titles), xlabel="Década", value_label="% títulos")


def _genre_quality(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.bubble(
        metrics.genre_quality(titles),
        x="titulos",
        y="imdb",
        size="calificados",
        text="genero",
        labels={"titulos": "Títulos en el catálogo", "imdb": "IMDb promedio", "calificados": "Títulos calificados"},
    )


def _imdb_by_decade(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.line(
        metrics.imdb_by_decade(titles),
        x="decada",
        y="imdb",
        hover=["titulos"],
        labels={"decada": "Década de estreno", "imdb": "IMDb promedio", "titulos": "Títulos"},
    )


def _runtime_by_decade(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.line(
        metrics.movie_runtime_by_decade(titles),
        x="decada",
        y="duracion",
        hover=["titulos"],
        labels={"decada": "Década de estreno", "duracion": "Duración mediana (min)", "titulos": "Películas"},
    )


def _popularity_vs_quality(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    data = metrics.popularity_vs_quality(titles)
    fig = figures.scatter(
        data,
        x="tmdb_popularity",
        y="imdb_score",
        color="tipo",
        hover_name="title",
        log_x=True,
        labels={"tmdb_popularity": "Popularidad TMDB (escala log)", "imdb_score": "IMDb"},
    )
    rho = metrics.rank_correlation(data, "tmdb_popularity", "imdb_score")
    return figures.note(fig, f"Correlación (Spearman) = {rho:.2f}") if rho is not None else fig


def _top_rated(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    fig = figures.bar_h(
        metrics.top_rated(titles),
        x="imdb_score",
        y="title",
        color="tipo",
        hover=["release_year", "imdb_votes"],
        labels={"imdb_score": "IMDb", "title": "", "release_year": "Año", "imdb_votes": "Votos", "tipo": ""},
    )
    return fig.update_xaxes(range=[7, 10])


def _seasons_quality(titles: pd.DataFrame, _: pd.DataFrame) -> go.Figure:
    return figures.box(
        metrics.imdb_by_seasons(titles),
        x="temporadas",
        y="imdb_score",
        hover_name="title",
        labels={"temporadas": "Temporadas", "imdb_score": "IMDb"},
    )


def _top_directors(titles: pd.DataFrame, credits: pd.DataFrame) -> go.Figure:
    return figures.bar_h(
        metrics.top_people(titles, credits, "DIRECTOR"),
        x="titulos",
        y="name",
        color="imdb",
        range_color=(5.5, 8.5),
        labels={"titulos": "Títulos", "name": "", "imdb": "IMDb prom."},
    )


def _top_actors(titles: pd.DataFrame, credits: pd.DataFrame) -> go.Figure:
    return figures.bar_h(
        metrics.top_people(titles, credits, "ACTOR"),
        x="titulos",
        y="name",
        color="pct_animacion",
        range_color=(0, 100),
        labels={"titulos": "Títulos", "name": "", "pct_animacion": "% animación"},
    )


CHARTS: tuple[ChartSpec, ...] = (
    ChartSpec(
        "g-decades",
        "Catálogo por década de estreno",
        "¿HBO Max es un catálogo de estrenos o de clásicos?",
        "El 30 % del catálogo se estrenó antes del año 2000 y hay películas desde la década de 1910: HBO Max mantiene una biblioteca de clásicos (Warner, TCM, Criterion) muy superior a la de otras plataformas. Las series prácticamente no existen antes de 1990 y su peso crece en cada década: en los 2020 ya son el 41 % de los estrenos.",
        _catalog_by_decade,
    ),
    ChartSpec(
        "g-genre-mix",
        "Peso de cada género por década (%)",
        "¿Cómo ha cambiado la oferta de géneros con el tiempo?",
        "El drama dominaba el cine clásico (70–80 % de los títulos entre 1930 y 1959) y hoy está en torno al 35 %. El romance cae del 51 % en los años 30 al 6 % en los 2020, mientras que el documental pasa de menos del 3 % entre 1930 y 1959 a más del 30 % desde 2010: la plataforma apuesta por la no ficción en su contenido reciente.",
        _genre_mix,
    ),
    ChartSpec(
        "g-genre-quality",
        "Cantidad vs. calidad por género",
        "¿Los géneros más abundantes son también los mejor valorados?",
        "Cantidad y calidad no van de la mano. Comedia (1.120 títulos) y acción están por debajo del promedio, y terror es el peor valorado (6,5). Los mejor puntuados son géneros de nicho: historia (7,5), documental y bélico (7,3–7,4). El drama es la excepción: es el género más abundante y aun así está entre los mejor valorados (7,15).",
        _genre_quality,
    ),
    ChartSpec(
        "g-imdb-decade",
        "IMDb promedio por década de estreno",
        "¿Las producciones antiguas están mejor valoradas que las recientes?",
        "Las producciones de 1920–1960 promedian ~7,4 en IMDb y las de 1980 en adelante ~6,8. No es que el cine antiguo fuera mejor: es un sesgo de supervivencia. Del pasado solo se conservan y licencian los clásicos que resistieron el tiempo, mientras que del presente llega todo lo que se produce.",
        _imdb_by_decade,
    ),
    ChartSpec(
        "g-runtime",
        "Duración mediana de las películas",
        "¿Las películas se han vuelto más largas o más cortas?",
        "La duración mediana de las películas se estabilizó en ~100–105 minutos desde los años 40 hasta los 2000, y en las dos últimas décadas bajó a ~90 minutos. Antes de 1930 dominan los cortometrajes y largometrajes mudos más breves (Chaplin, por ejemplo).",
        _runtime_by_decade,
    ),
    ChartSpec(
        "g-popularity",
        "Popularidad vs. calidad",
        "¿Lo más popular es lo mejor valorado?",
        "La correlación de rangos entre popularidad en TMDB y puntuación IMDb es prácticamente nula (≈ 0,01). Hay títulos muy populares con notas mediocres y joyas con poca audiencia: la popularidad no es un buen indicador de calidad para recomendar contenido.",
        _popularity_vs_quality,
    ),
    ChartSpec(
        "g-top-rated",
        "Los 10 títulos mejor valorados (≥10.000 votos)",
        "¿Qué es lo mejor del catálogo y de dónde viene?",
        "7 de los 10 títulos mejor valorados son series, y cinco de ellas son producciones originales de HBO (Chernobyl, Band of Brothers, The Wire, Game of Thrones, The Sopranos). El valor diferencial de la plataforma está en su contenido propio, no solo en el licenciado.",
        _top_rated,
    ),
    ChartSpec(
        "g-seasons",
        "IMDb de las series según sus temporadas",
        "¿Las series que duran más temporadas son mejores?",
        "Las series con 3–5 temporadas tienen una mediana de IMDb ligeramente mayor (7,8–7,95) que las de una sola temporada (7,6). El efecto es pequeño y se explica por selección: solo se renuevan las series que funcionan. A partir de 6 temporadas la nota vuelve a bajar un poco.",
        _seasons_quality,
    ),
    ChartSpec(
        "g-directors",
        "Directores con más títulos",
        "¿Qué directores definen el catálogo y qué tan bien valorados están?",
        "Los directores más presentes son autores clásicos: Charlie Chaplin, Yasujirō Ozu, Satyajit Ray, Ingmar Bergman y Hayao Miyazaki, todos con IMDb promedio cercano a 8. Junto a ellos aparecen documentalistas de HBO (Jon Alpert, Alexandra Pelosi) y Sam Liu, director de las películas animadas de DC.",
        _top_directors,
    ),
    ChartSpec(
        "g-actors",
        "Actores con más títulos",
        "¿Quiénes aparecen más en el catálogo y en qué tipo de contenido?",
        "Los actores con más títulos son actores de doblaje (Grey DeLisle, Frank Welker, Tara Strong, Tom Kenny) y más de la mitad de sus títulos son animados: el catálogo de animación de Warner Bros. y Cartoon Network pesa más que cualquier estrella de cine. El único actor de imagen real en el top es Charlie Chaplin.",
        _top_actors,
    ),
)

In [ ]:
%%writefile hbo_max_dashboard/views/components.py
"""Componentes HTML reutilizables del dashboard."""

from dash import dcc, html

from .charts import ChartSpec
from .kpis import KpiSpec
from .theme import CARD, MUTED, PRIMARY

CARD_STYLE = {"background": CARD, "borderRadius": "10px", "padding": "12px 16px"}


def grid(children: list, min_width: int, **style) -> html.Div:
    return html.Div(
        children,
        style={
            "display": "grid",
            "gridTemplateColumns": f"repeat(auto-fit, minmax(min({min_width}px, 100%), 1fr))",
            "gap": "12px",
            "marginBottom": "12px",
            **style,
        },
    )


def filter_card(label: str, control) -> html.Div:
    return html.Div([html.Div(label, style={"marginBottom": "8px", "fontWeight": 600}), control], style=CARD_STYLE)


def kpi_card(spec: KpiSpec) -> html.Div:
    return html.Div(
        [
            html.Div(spec.label, style={"color": MUTED, "fontSize": "13px"}),
            html.Div(id=spec.id, style={"color": PRIMARY, "fontSize": "28px", "fontWeight": 700}),
        ],
        style={**CARD_STYLE, "textAlign": "center"},
    )


def chart_card(spec: ChartSpec) -> html.Div:
    return html.Div(
        [
            dcc.Graph(id=spec.id, config={"displaylogo": False}),
            html.Div(
                spec.question, style={"color": MUTED, "fontSize": "13px", "fontStyle": "italic", "marginTop": "6px"}
            ),
        ],
        style=CARD_STYLE,
    )

In [ ]:
%%writefile hbo_max_dashboard/views/figures.py
"""Primitivas de gráficas Plotly con el tema del dashboard.

Son funciones de presentación puras: reciben datos ya agregados y no conocen el dominio.
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from .theme import CARD, FONT, GRID, MUTED, PALETTE, PRIMARY, SEQUENTIAL, TEXT

HEIGHT = 380


def style(fig: go.Figure, title: str = "") -> go.Figure:
    fig.update_layout(
        title=dict(text=title, font=dict(size=15, color=TEXT), x=0.02, y=0.97),
        paper_bgcolor=CARD,
        plot_bgcolor=CARD,
        font=dict(color=TEXT, family=FONT),
        margin=dict(l=10, r=10, t=50, b=10),
        legend=dict(title=None, orientation="h", y=1.0, yanchor="bottom", x=1, xanchor="right"),
        hoverlabel=dict(font=dict(family=FONT)),
        height=HEIGHT,
    )
    fig.update_xaxes(gridcolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig


def empty(message: str = "Sin datos para esta selección") -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(text=message, showarrow=False, font=dict(color=MUTED, size=14))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return style(fig)


def note(fig: go.Figure, text: str) -> go.Figure:
    """Anotación pequeña en la esquina inferior derecha (p. ej. una correlación)."""
    fig.add_annotation(
        text=text,
        xref="paper",
        yref="paper",
        x=1,
        y=0,
        xanchor="right",
        yanchor="bottom",
        showarrow=False,
        font=dict(color=MUTED, size=12),
        bgcolor=CARD,
    )
    return fig


def stacked_bar(data: pd.DataFrame, x: str, y: str, color: str, labels: dict[str, str]) -> go.Figure:
    if data.empty:
        return empty()
    fig = px.bar(data, x=x, y=y, color=color, color_discrete_map=PALETTE, labels=labels)
    fig.update_layout(barmode="stack")
    return style(fig)


def heatmap(pivot: pd.DataFrame, xlabel: str, value_label: str) -> go.Figure:
    if pivot.empty:
        return empty()
    fig = px.imshow(
        pivot,
        text_auto=".0f",
        aspect="auto",
        color_continuous_scale=SEQUENTIAL,
        labels=dict(x=xlabel, y="", color=value_label),
    )
    fig.update_xaxes(type="category", showgrid=False)
    fig.update_yaxes(showgrid=False)
    return style(fig)


def bubble(data: pd.DataFrame, x: str, y: str, size: str, text: str, labels: dict[str, str]) -> go.Figure:
    if data.empty:
        return empty()
    fig = px.scatter(
        data,
        x=x,
        y=y,
        size=size,
        text=text,
        labels=labels,
        size_max=40,
        color=y,
        color_continuous_scale=SEQUENTIAL[1:],
        hover_name=text,
    )
    fig.update_traces(textposition="top center", textfont=dict(size=11, color=TEXT))
    fig.update_layout(coloraxis_showscale=False)
    return style(fig)


def line(data: pd.DataFrame, x: str, y: str, labels: dict[str, str], hover: list[str] | None = None) -> go.Figure:
    if data.empty:
        return empty()
    fig = px.line(data, x=x, y=y, labels=labels, markers=True, hover_data=hover, color_discrete_sequence=[PRIMARY])
    fig.update_traces(line=dict(width=3), marker=dict(size=8))
    return style(fig)


def scatter(
    data: pd.DataFrame,
    x: str,
    y: str,
    color: str,
    labels: dict[str, str],
    hover_name: str,
    log_x: bool = False,
) -> go.Figure:
    if data.empty:
        return empty()
    fig = px.scatter(
        data,
        x=x,
        y=y,
        color=color,
        color_discrete_map=PALETTE,
        labels=labels,
        hover_name=hover_name,
        log_x=log_x,
        opacity=0.55,
    )
    fig.update_traces(marker=dict(size=6))
    return style(fig)


def bar_h(
    data: pd.DataFrame,
    x: str,
    y: str,
    labels: dict[str, str],
    color: str | None = None,
    hover: list[str] | None = None,
    range_color: tuple[float, float] | None = None,
) -> go.Figure:
    """Barras horizontales con el primer elemento arriba. `color` puede ser categórico o numérico."""
    if data.empty:
        return empty()
    data = data.iloc[::-1]
    if color and pd.api.types.is_numeric_dtype(data[color]):
        fig = px.bar(
            data,
            x=x,
            y=y,
            orientation="h",
            labels=labels,
            hover_data=hover,
            color=color,
            color_continuous_scale=SEQUENTIAL[1:],
            range_color=range_color,
        )
    else:
        fig = px.bar(
            data,
            x=x,
            y=y,
            orientation="h",
            labels=labels,
            hover_data=hover,
            color=color,
            color_discrete_map=PALETTE,
            color_discrete_sequence=[PRIMARY],
        )
    fig.update_yaxes(title=None)
    return style(fig)


def box(data: pd.DataFrame, x: str, y: str, labels: dict[str, str], hover_name: str) -> go.Figure:
    if data.empty:
        return empty()
    fig = px.box(
        data,
        x=x,
        y=y,
        labels=labels,
        points="all",
        hover_name=hover_name,
        color_discrete_sequence=[PRIMARY],
    )
    fig.update_traces(marker=dict(size=4, opacity=0.5), jitter=0.4)
    return style(fig)

In [ ]:
%%writefile hbo_max_dashboard/views/kpis.py
"""Indicadores de la cabecera: qué se muestra y cómo se formatea."""

from collections.abc import Callable
from dataclasses import dataclass

from ..logic.metrics import Kpis

NO_DATA = "–"


def fmt_int(value: int) -> str:
    """Separador de miles en español: 3294 -> '3.294'."""
    return f"{value:,}".replace(",", ".")


def fmt_pct(value: float | None) -> str:
    return NO_DATA if value is None else f"{value * 100:.0f} %"


def fmt_score(value: float | None) -> str:
    return NO_DATA if value is None else f"{value:.2f}".replace(".", ",")


@dataclass(frozen=True)
class KpiSpec:
    id: str
    label: str
    format: Callable[[Kpis], str]


KPIS: tuple[KpiSpec, ...] = (
    KpiSpec("k-total", "Títulos", lambda k: fmt_int(k.total)),
    KpiSpec("k-movies", "Películas", lambda k: fmt_int(k.movies)),
    KpiSpec("k-shows", "Series", lambda k: fmt_int(k.shows)),
    KpiSpec("k-imdb", "IMDb promedio", lambda k: fmt_score(k.imdb_mean)),
    KpiSpec("k-classics", "Estrenados antes de 2000", lambda k: fmt_pct(k.share_before_2000)),
)

In [ ]:
%%writefile hbo_max_dashboard/views/layout.py
"""Estructura de la página. No contiene lógica de datos."""

from dash import dcc, html

from ..config import ALL_GENRES, TYPE_LABELS
from .charts import CHARTS
from .components import chart_card, filter_card, grid, kpi_card
from .kpis import KPIS
from .theme import BG, FONT, MUTED, PRIMARY, TEXT

FILTER_TYPE, FILTER_GENRE, FILTER_YEARS = "f-type", "f-genre", "f-years"


def _filters(genres: list[str], year_bounds: tuple[int, int]) -> html.Div:
    year_min, year_max = year_bounds
    first_mark = year_min - year_min % 10 + 10
    return grid(
        [
            filter_card(
                "Tipo",
                dcc.Checklist(
                    id=FILTER_TYPE,
                    options=list(TYPE_LABELS.values()),
                    value=list(TYPE_LABELS.values()),
                    inline=True,
                    inputStyle={"marginRight": "4px", "marginLeft": "12px"},
                ),
            ),
            filter_card(
                "Género",
                dcc.Dropdown(
                    id=FILTER_GENRE,
                    value=ALL_GENRES,
                    clearable=False,
                    style={"color": "#000"},
                    options=[{"label": "Todos los géneros", "value": ALL_GENRES}]
                    + [{"label": g.capitalize(), "value": g} for g in genres],
                ),
            ),
            filter_card(
                "Año de estreno",
                dcc.RangeSlider(
                    id=FILTER_YEARS,
                    min=year_min,
                    max=year_max,
                    step=1,
                    value=[year_min, year_max],
                    marks={y: str(y) for y in range(first_mark, year_max + 1, 20)},
                    tooltip={"placement": "bottom"},
                ),
            ),
        ],
        min_width=260,
    )


def build_layout(genres: list[str], year_bounds: tuple[int, int]) -> html.Div:
    return html.Div(
        style={"background": BG, "color": TEXT, "minHeight": "100vh", "padding": "20px", "fontFamily": FONT},
        children=[
            html.H1("HBO MAX · Películas y Series", style={"color": PRIMARY, "margin": 0}),
            html.P(
                "Catálogo de HBO Max en EE. UU. (mayo 2022): qué ofrece, de qué épocas y qué tan bien valorado está.",
                style={"color": MUTED},
            ),
            _filters(genres, year_bounds),
            grid([kpi_card(spec) for spec in KPIS], min_width=180),
            grid([chart_card(spec) for spec in CHARTS], min_width=520),
        ],
    )

In [ ]:
%%writefile hbo_max_dashboard/views/theme.py
"""Paleta y estilos visuales compartidos por las vistas."""

from ..config import MOVIE, SHOW

PRIMARY = "#9B5CF6"
SECONDARY = "#2DD4BF"
PALETTE = {MOVIE: PRIMARY, SHOW: SECONDARY}
SEQUENTIAL = ["#1b1230", "#4c2a85", "#7c4ddb", "#b794f6", "#ede4ff"]

BG = "#10091c"
CARD = "#1b1230"
GRID = "#2e2447"
TEXT = "#f2f0f7"
MUTED = "#9d93b5"
FONT = "system-ui, -apple-system, Segoe UI, sans-serif"

## 4. Procesamiento de los datos
### Datos crudos

In [ ]:
from hbo_max_dashboard.data.cleaning import missing_report
from hbo_max_dashboard.data.repository import read_csv

raw_titles = read_csv("titles", DATA_DIR)
raw_credits = read_csv("credits", DATA_DIR)
print(f"titles: {raw_titles.shape}   credits: {raw_credits.shape}")
raw_titles.head(3)

### Valores faltantes

In [ ]:
missing_report(raw_titles)

### Limpieza aplicada (`data/cleaning.py`)
- `genres` y `production_countries` vienen como texto (`"['drama', 'crime']"`): se convierten en listas y los géneros se traducen al español.
- `type` (MOVIE/SHOW) se traduce a `tipo` (Película/Serie) y se deriva la `decada` de estreno.
- `age_certification` faltante se marca como `Sin dato`.
- Se eliminan títulos y créditos duplicados.
- Las puntuaciones IMDb solo se promedian para títulos con al menos 1.000 votos, y los grupos con menos de 10 títulos (o décadas con menos de 20) se omiten para evitar promedios poco confiables.

In [ ]:
from hbo_max_dashboard.data import load_catalog

catalog = load_catalog(DATA_DIR)
print(f"Títulos: {len(catalog.titles)} (duplicados eliminados: {len(raw_titles) - len(catalog.titles)})")
print(f"Créditos: {len(catalog.credits)} (duplicados eliminados: {len(raw_credits) - len(catalog.credits)})")
catalog.titles[["title", "tipo", "release_year", "decada", "genres", "imdb_score"]].head()

## 5. Análisis: preguntas y hallazgos
Cada gráfica responde una pregunta sobre el catálogo completo.

In [ ]:
from hbo_max_dashboard.views.charts import CHARTS


def show(chart_id: str) -> None:
    spec = next(c for c in CHARTS if c.id == chart_id)
    spec.render(catalog.titles, catalog.credits).show()

### 5.1 Catálogo por década de estreno
**Pregunta:** ¿HBO Max es un catálogo de estrenos o de clásicos?

In [ ]:
show("g-decades")

**Hallazgo:** El 30 % del catálogo se estrenó antes del año 2000 y hay películas desde la década de 1910: HBO Max mantiene una biblioteca de clásicos (Warner, TCM, Criterion) muy superior a la de otras plataformas. Las series prácticamente no existen antes de 1990 y su peso crece en cada década: en los 2020 ya son el 41 % de los estrenos.

### 5.2 Peso de cada género por década (%)
**Pregunta:** ¿Cómo ha cambiado la oferta de géneros con el tiempo?

In [ ]:
show("g-genre-mix")

**Hallazgo:** El drama dominaba el cine clásico (70–80 % de los títulos entre 1930 y 1959) y hoy está en torno al 35 %. El romance cae del 51 % en los años 30 al 6 % en los 2020, mientras que el documental pasa de menos del 3 % entre 1930 y 1959 a más del 30 % desde 2010: la plataforma apuesta por la no ficción en su contenido reciente.

### 5.3 Cantidad vs. calidad por género
**Pregunta:** ¿Los géneros más abundantes son también los mejor valorados?

In [ ]:
show("g-genre-quality")

**Hallazgo:** Cantidad y calidad no van de la mano. Comedia (1.120 títulos) y acción están por debajo del promedio, y terror es el peor valorado (6,5). Los mejor puntuados son géneros de nicho: historia (7,5), documental y bélico (7,3–7,4). El drama es la excepción: es el género más abundante y aun así está entre los mejor valorados (7,15).

### 5.4 IMDb promedio por década de estreno
**Pregunta:** ¿Las producciones antiguas están mejor valoradas que las recientes?

In [ ]:
show("g-imdb-decade")

**Hallazgo:** Las producciones de 1920–1960 promedian ~7,4 en IMDb y las de 1980 en adelante ~6,8. No es que el cine antiguo fuera mejor: es un sesgo de supervivencia. Del pasado solo se conservan y licencian los clásicos que resistieron el tiempo, mientras que del presente llega todo lo que se produce.

### 5.5 Duración mediana de las películas
**Pregunta:** ¿Las películas se han vuelto más largas o más cortas?

In [ ]:
show("g-runtime")

**Hallazgo:** La duración mediana de las películas se estabilizó en ~100–105 minutos desde los años 40 hasta los 2000, y en las dos últimas décadas bajó a ~90 minutos. Antes de 1930 dominan los cortometrajes y largometrajes mudos más breves (Chaplin, por ejemplo).

### 5.6 Popularidad vs. calidad
**Pregunta:** ¿Lo más popular es lo mejor valorado?

In [ ]:
show("g-popularity")

**Hallazgo:** La correlación de rangos entre popularidad en TMDB y puntuación IMDb es prácticamente nula (≈ 0,01). Hay títulos muy populares con notas mediocres y joyas con poca audiencia: la popularidad no es un buen indicador de calidad para recomendar contenido.

### 5.7 Los 10 títulos mejor valorados (≥10.000 votos)
**Pregunta:** ¿Qué es lo mejor del catálogo y de dónde viene?

In [ ]:
show("g-top-rated")

**Hallazgo:** 7 de los 10 títulos mejor valorados son series, y cinco de ellas son producciones originales de HBO (Chernobyl, Band of Brothers, The Wire, Game of Thrones, The Sopranos). El valor diferencial de la plataforma está en su contenido propio, no solo en el licenciado.

### 5.8 IMDb de las series según sus temporadas
**Pregunta:** ¿Las series que duran más temporadas son mejores?

In [ ]:
show("g-seasons")

**Hallazgo:** Las series con 3–5 temporadas tienen una mediana de IMDb ligeramente mayor (7,8–7,95) que las de una sola temporada (7,6). El efecto es pequeño y se explica por selección: solo se renuevan las series que funcionan. A partir de 6 temporadas la nota vuelve a bajar un poco.

### 5.9 Directores con más títulos
**Pregunta:** ¿Qué directores definen el catálogo y qué tan bien valorados están?

In [ ]:
show("g-directors")

**Hallazgo:** Los directores más presentes son autores clásicos: Charlie Chaplin, Yasujirō Ozu, Satyajit Ray, Ingmar Bergman y Hayao Miyazaki, todos con IMDb promedio cercano a 8. Junto a ellos aparecen documentalistas de HBO (Jon Alpert, Alexandra Pelosi) y Sam Liu, director de las películas animadas de DC.

### 5.10 Actores con más títulos
**Pregunta:** ¿Quiénes aparecen más en el catálogo y en qué tipo de contenido?

In [ ]:
show("g-actors")

**Hallazgo:** Los actores con más títulos son actores de doblaje (Grey DeLisle, Frank Welker, Tara Strong, Tom Kenny) y más de la mitad de sus títulos son animados: el catálogo de animación de Warner Bros. y Cartoon Network pesa más que cualquier estrella de cine. El único actor de imagen real en el top es Charlie Chaplin.

## 6. Conclusiones
- HBO Max combina dos catálogos: una biblioteca de clásicos (30 % anterior al año 2000) y una oferta reciente cada vez más centrada en series y documentales.
- La calidad percibida depende más del nicho que del volumen: historia, documental y bélico superan a comedia, acción y terror.
- Las producciones antiguas puntúan más alto por sesgo de supervivencia, no porque el cine antiguo sea mejor en sí mismo.
- La popularidad no predice la calidad, y lo mejor valorado de la plataforma son sus series originales de HBO.
- Por número de apariciones, el catálogo lo definen los autores clásicos y los actores de doblaje de animación, no las estrellas de Hollywood.

## 7. Dashboard interactivo
Los filtros de tipo, género y año recalculan los KPIs y las 10 gráficas.

In [ ]:
from hbo_max_dashboard.app import create_app

app = create_app(DATA_DIR)
app.run(jupyter_mode="inline", height=2600, debug=False)